# Experimento de pérdida: suavizado de etiquetas

In [ ]:
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Activa T4 GPU en Entorno de ejecución → Cambiar tipo de entorno."
DEVICE = torch.device("cuda")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/PuertoPrisma")
ARCHIVE = PROJECT / "cartoonset64.zip"
assert ARCHIVE.is_file(), f"No se encontró {ARCHIVE}. Revisa los archivos de Drive."
RUN = PROJECT / "runs/loss_label_smoothing"
RUN.mkdir(parents=True, exist_ok=True)
probe = RUN / ".write_check"
probe.write_text("ok")
assert probe.read_text() == "ok"
probe.unlink()
print("Datos encontrados. Guardaremos el entrenamiento en:", RUN)

## Diseño del experimento

In [ ]:
import random, json, time, os, hashlib, zipfile, shutil, sys
import numpy as np
import PIL
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader

CONFIG = dict(seed=2026, epochs=30, batch_size=128, latent_dim=100,
              image_size=64, channels=3, features=64, lr=0.0002,
              beta1=0.5, beta2=0.999, loss="bce_non_saturating_real_label_0.9",
              stabilization="baseline_batchnorm", augmentation="none")
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

LOCAL = Path("/content/puerto_prisma_train")
DATA = LOCAL / "cartoonset64"
LOCAL.mkdir(parents=True, exist_ok=True)
local_zip = LOCAL / "cartoonset64.zip"
print("Copiando datos de Drive al disco local...")
shutil.copy2(ARCHIVE, local_zip)
digest = hashlib.sha256(local_zip.read_bytes()).hexdigest()
with zipfile.ZipFile(local_zip) as bundle:
    names = bundle.namelist()
    assert all(not Path(n).is_absolute() and ".." not in Path(n).parts for n in names)
    assert sum(n.endswith(".png") for n in names) == 10000
    assert bundle.testzip() is None, "ZIP dañado"
    if not DATA.exists():
        DATA.mkdir()
        bundle.extractall(DATA)

class CartoonDataset(Dataset):
    def __init__(self, folder):
        self.files = sorted(Path(folder).rglob("*.png"))
        assert len(self.files) == 10000
    def __len__(self):
        return len(self.files)
    def __getitem__(self, index):
        with Image.open(self.files[index]) as image:
            assert image.size == (64, 64)
            array = np.asarray(image.convert("RGB"), dtype=np.float32).copy()
        return torch.from_numpy(array).permute(2, 0, 1) / 127.5 - 1.0

dataset = CartoonDataset(DATA)
VERSIONS = dict(python=sys.version, torch=torch.__version__, numpy=np.__version__,
                pillow=PIL.__version__, cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0))
print("Imágenes:", len(dataset))
print("Configuración:", CONFIG)
print("Versiones:", VERSIONS)
HYPOTHESIS = 'Creemos que usar 0.9 como etiqueta de las imágenes reales mejorará la calidad visual porque reduce el incentivo de D a producir confianza extrema para los datos reales. Lo consideraremos útil si las rejillas de ruido fijo muestran menos artefactos y mantienen rostros distintos respecto a la base, sin una pérdida persistente de estructura.'
CHANGE = 'Solo cambia la etiqueta de las imágenes reales en la BCE de D: 1.0 → 0.9. Las falsas conservan 0.0 y G conserva 1.0. No se modifica arquitectura ni tasa de aprendizaje.'


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, features*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features*8), nn.ReLU(True),
            nn.ConvTranspose2d(features*8, features*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*4), nn.ReLU(True),
            nn.ConvTranspose2d(features*4, features*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*2), nn.ReLU(True),
            nn.ConvTranspose2d(features*2, features, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features), nn.ReLU(True),
            nn.ConvTranspose2d(features, 3, 4, 2, 1, bias=False), nn.Tanh())
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, features, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(features, features*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(features*2, features*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(features*4, features*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(features*8, 1, 4, 1, 0, bias=False))
    def forward(self, image):
        return self.net(image).reshape(-1)

def initialize(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, 0.0, 0.02)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.normal_(module.weight, 1.0, 0.02)
        nn.init.zeros_(module.bias)

G = Generator(CONFIG["latent_dim"], CONFIG["features"]).to(DEVICE)
D = Discriminator(CONFIG["features"]).to(DEVICE)
G.apply(initialize)
D.apply(initialize)
optimizer_g = torch.optim.Adam(G.parameters(), lr=CONFIG["lr"], betas=(CONFIG["beta1"], CONFIG["beta2"]))
optimizer_d = torch.optim.Adam(D.parameters(), lr=CONFIG["lr"], betas=(CONFIG["beta1"], CONFIG["beta2"]))
criterion = nn.BCEWithLogitsLoss()
# Generador aleatorio separado para que el ruido de evaluación no altere el entrenamiento.
fixed_rng = torch.Generator().manual_seed(CONFIG["seed"] + 1)
fixed_z = torch.randn(16, CONFIG["latent_dim"], 1, 1, generator=fixed_rng).to(DEVICE)
print("Parámetros G:", sum(p.numel() for p in G.parameters()))
print("Parámetros D:", sum(p.numel() for p in D.parameters()))

## Evidencia y recuperación

In [ ]:
def atomic_json(value, destination):
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.write_text(json.dumps(value, indent=2))
    temporary.replace(destination)

def save_checkpoint(epoch, history):
    state = dict(epoch=epoch, config=CONFIG, data_sha256=digest, versions=VERSIONS,
                 generator=G.state_dict(), discriminator=D.state_dict(),
                 optimizer_g=optimizer_g.state_dict(), optimizer_d=optimizer_d.state_dict(),
                 fixed_z=fixed_z.cpu(), history=history, python_rng=random.getstate(),
                 numpy_rng=np.random.get_state(), torch_rng=torch.get_rng_state(),
                 cuda_rng=torch.cuda.get_rng_state_all())
    destination = RUN / f"checkpoint_epoch_{epoch:03d}.pt"
    temporary = destination.with_suffix(".pt.part")
    torch.save(state, temporary)
    temporary.replace(destination)

def save_samples(epoch):
    was_training = G.training
    G.eval()
    with torch.no_grad():
        samples = ((G(fixed_z).cpu().clamp(-1, 1) + 1) / 2 * 255).round().byte()
    G.train(was_training)
    sheet = Image.new("RGB", (64*4, 64*4), "white")
    for i, sample in enumerate(samples):
        sheet.paste(Image.fromarray(sample.permute(1,2,0).numpy()), ((i%4)*64, (i//4)*64))
    sheet.save(RUN / f"fixed_epoch_{epoch:03d}.png")
    return sheet

atomic_json(dict(hypothesis=HYPOTHESIS, change=CHANGE, baseline="base_bce", status="written_before_training"), RUN / "hypothesis.json")

history = []
start_epoch = 0
checkpoints = sorted(RUN.glob("checkpoint_epoch_*.pt"))
if checkpoints:
    # Carga exclusivamente checkpoints que generamos nosotros, no archivos externos.
    state = torch.load(checkpoints[-1], map_location="cpu", weights_only=False)
    assert state["config"] == CONFIG, "Configuración distinta: usa una carpeta nueva."
    assert state["data_sha256"] == digest, "Los datos difieren de la ejecución guardada."
    G.load_state_dict(state["generator"])
    D.load_state_dict(state["discriminator"])
    optimizer_g.load_state_dict(state["optimizer_g"])
    optimizer_d.load_state_dict(state["optimizer_d"])
    fixed_z = state["fixed_z"].to(DEVICE)
    history = state["history"]
    start_epoch = state["epoch"]
    random.setstate(state["python_rng"])
    np.random.set_state(state["numpy_rng"])
    torch.set_rng_state(state["torch_rng"])
    torch.cuda.set_rng_state_all(state["cuda_rng"])
    print("Retomando después de la época", start_epoch)
else:
    atomic_json(dict(config=CONFIG, versions=VERSIONS, data_sha256=digest), RUN / "config.json")
    torch.save(fixed_z.cpu(), RUN / "fixed_z.pt")
    save_samples(0)
    save_checkpoint(0, history)
    print("Nueva ejecución. Ruido fijo y checkpoint inicial guardados.")

## Entrenar la base

In [ ]:
from IPython.display import display
G.train()
D.train()
for epoch in range(start_epoch + 1, CONFIG["epochs"] + 1):
    started = time.perf_counter()
    # Orden reproducible por época, independiente del generador de ruido.
    batch_rng = torch.Generator().manual_seed(CONFIG["seed"] + epoch)
    loader = DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=True,
                        generator=batch_rng, num_workers=2, pin_memory=True, drop_last=False)
    totals = dict(g_loss=0.0, d_loss=0.0, d_real=0.0, d_fake=0.0)
    seen = 0
    for real in loader:
        real = real.to(DEVICE, non_blocking=True)
        batch = real.size(0)
        # D: aprende a distinguir imágenes reales y generadas.
        D.requires_grad_(True)
        optimizer_d.zero_grad(set_to_none=True)
        z = torch.randn(batch, CONFIG["latent_dim"], 1, 1, device=DEVICE)
        fake = G(z)
        real_logits = D(real)
        fake_logits = D(fake.detach())
        d_loss = criterion(real_logits, torch.full_like(real_logits, 0.9)) + criterion(fake_logits, torch.zeros_like(fake_logits))
        d_loss.backward()
        optimizer_d.step()
        # G: pérdida no saturante, busca que D clasifique las imágenes como reales.
        D.requires_grad_(False)
        optimizer_g.zero_grad(set_to_none=True)
        g_logits = D(fake)
        g_loss = criterion(g_logits, torch.ones_like(g_logits))
        g_loss.backward()
        optimizer_g.step()
        D.requires_grad_(True)
        values = dict(g_loss=g_loss.item(), d_loss=d_loss.item(),
                      d_real=real_logits.detach().sigmoid().mean().item(),
                      d_fake=fake_logits.detach().sigmoid().mean().item())
        for key, value in values.items():
            totals[key] += value * batch
        seen += batch
    torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    row = dict(epoch=epoch, seconds=seconds, **{k:v/seen for k,v in totals.items()})
    history.append(row)
    sheet = save_samples(epoch)
    save_checkpoint(epoch, history)
    atomic_json(history, RUN / "history.json")
    print(f"Época {epoch}/{CONFIG['epochs']} guardada | G={row['g_loss']:.3f} D={row['d_loss']:.3f} | {seconds:.1f}s | restantes aprox. {seconds*(CONFIG['epochs']-epoch)/60:.1f} min", flush=True)
    if epoch == 1 or epoch % 5 == 0 or epoch == CONFIG["epochs"]:
        display(sheet.resize((512,512), Image.Resampling.NEAREST))
print("EXPERIMENTO TERMINADO. Artefactos en:", RUN)

## Curvas y evolución

In [ ]:
import matplotlib.pyplot as plt
assert history, "Primero completa al menos una época."
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for key, label in [("g_loss", "G"), ("d_loss", "D (suma real + fake)")]:
    axes[0].plot(epochs, [row[key] for row in history], label=label)
axes[0].set(xlabel="Época", ylabel="Pérdida BCE", title="Experimento de pérdida: suavizado de etiquetas")
axes[0].legend()
for key, label in [("d_real", "D(real)"), ("d_fake", "D(fake) antes de actualizar D")]:
    axes[1].plot(epochs, [row[key] for row in history], label=label)
axes[1].set(xlabel="Época", ylabel="Probabilidad media", ylim=(0,1))
axes[1].legend()
fig.tight_layout()
fig.savefig(RUN / "loss_curves.png", dpi=160)
plt.show()
points = sorted(set([0, 1, max(1, history[-1]["epoch"]//2), history[-1]["epoch"]]))
fig, axes = plt.subplots(1, len(points), figsize=(4*len(points),4))
for axis, epoch in zip(np.atleast_1d(axes), points):
    image = RUN / f"fixed_epoch_{epoch:03d}.png"
    if image.exists():
        with Image.open(image) as im:
            axis.imshow(im.copy())
    axis.set_title(f"Época {epoch}")
    axis.axis("off")
fig.tight_layout()
fig.savefig(RUN / "fixed_evolution.png", dpi=160)
plt.show()
print("Figuras guardadas en Drive. Comparte fixed_evolution.png y loss_curves.png para analizar la base.")

## Registrar el resultado